# V6 — Florence-2-base · H1 (Bootstrap Colab T4)

Notebook **base** del hito H1 del plan V6. Solo carga el modelo, el dataset y verifica el smoke test zero-shot. **No entrena** (eso es H3).

Plan: `Documentacion/plan_v6.md`. Decisiones: `memory/bot/decisions.md`.

**Stack** elegido para Florence-2 + T4 16 GB (NO derivado de V5):
- `transformers>=4.41,<4.46` (>=4.46 rompe `trust_remote_code` de Florence-2)
- `accelerate`, `peft`, `einops`, `timm`, `datasets`
- **fp16** (T4 compute 7.5 no soporta bf16 eficiente)
- **Sin Unsloth, sin flash-attn** (T4 no soporta FA2)

**Tarea**: tag custom `<EXTRACT_TOTAL>`. Output esperado tras H3: `total<loc_x1><loc_y1><loc_x2><loc_y2>`.

## A · Check entorno

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv,noheader
import torch
print('torch:', torch.__version__, '| cuda:', torch.version.cuda, '| available:', torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'compute capability: {cap}  → fp16 {"sí" if cap[0]<8 else "posible"}, bf16 nativo {"no" if cap[0]<8 else "sí"}')

## B · Instalación de dependencias

NO reinstalar torch (la imagen base de Colab ya trae uno compatible con T4 + CUDA 12). NO instalar flash-attn.

In [ ]:
!pip install -q \
    "transformers>=4.41,<4.46" \
    "accelerate>=0.30" \
    "peft>=0.11" \
    "einops>=0.8" \
    "timm>=0.9" \
    "datasets>=2.19" \
    "huggingface_hub>=0.24" \
    "pillow>=10"

## C · Mount Drive (preparado para checkpoints de H3)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/TFG/V6_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints futuros →', CKPT_DIR)

## D · Login HF (token desde Colab Secrets)

Crear el secret `HF_TOKEN` en Colab (panel izquierdo → 🔑) con un token **write** del usuario `Lacax`.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print('HF login ok')

## E · Cargar Florence-2-base en fp16

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = 'microsoft/Florence-2-base'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to('cuda').eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
vram = torch.cuda.memory_allocated() / 1e9
print(f'Florence-2-base loaded · params≈{n_params:.1f}M · fp16 VRAM≈{vram:.2f} GB')

## F · Cargar dataset Lacax/Tickets-total

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from PIL import Image
import json, io
import matplotlib.pyplot as plt

REPO = 'Lacax/Tickets-total'

def load_split(split: str):
    """Lee el JSONL del split desde el repo y devuelve list[dict]."""
    path = hf_hub_download(REPO, f'dataset_total_{split}.jsonl', repo_type='dataset')
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

def load_image(image_name: str, labeled: bool = False):
    """original/ -> entrada al modelo (sin marcas).
    etiquetadas/ -> solo verificacion visual humana (NO usar en training: data leakage)."""
    folder = 'etiquetadas' if labeled else 'original'
    path = hf_hub_download(REPO, f'{folder}/{image_name}', repo_type='dataset')
    return Image.open(path).convert('RGB')

train = load_split('train')
val   = load_split('val')
test  = load_split('test')
print(f'splits -> train={len(train)}  val={len(val)}  test={len(test)}')
print('ejemplo val[0]:', val[0])

# Verificacion visual: original (lo que ve el modelo) vs etiquetada (con bbox rojo)
sample = val[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 7))
axes[0].imshow(load_image(sample['image_path'], labeled=False))
axes[0].set_title('original (input training)'); axes[0].axis('off')
axes[1].imshow(load_image(sample['image_path'], labeled=True))
axes[1].set_title(f"etiquetada (verificacion) - total={sample['total']}"); axes[1].axis('off')
plt.tight_layout(); plt.show()


## G · Añadir tag custom `<EXTRACT_TOTAL>`

Florence-2 admite tags custom siempre que se redimensione el embedding. La tarea se aprenderá en H3 desde cero, pero el setup vive aquí.

In [ ]:
EXTRACT_TAG = '<EXTRACT_TOTAL>'
added = processor.tokenizer.add_tokens([EXTRACT_TAG], special_tokens=True)
print(f'añadidos {added} tokens nuevos (0 si ya existía)')
model.resize_token_embeddings(len(processor.tokenizer))
tag_ids = processor.tokenizer(EXTRACT_TAG, add_special_tokens=False).input_ids
print(f"Tag '{EXTRACT_TAG}' tokenized to ids={tag_ids}  (debería ser 1 id único)")

## H · Smoke test zero-shot

Sin entrenar el tag, la salida será ruidosa (no vamos a evaluar correctitud). **El criterio es no-OOM y < 12 GB de VRAM** para dejar margen al training en H3.

In [ ]:
sample = val[0]
img = load_image(sample['image_path'])  # labeled=False -> original sin marcas
print('imagen:', sample['image_path'], 'size:', img.size, 'total GT:', sample['total'])

inputs = processor(text=EXTRACT_TAG, images=img, return_tensors='pt').to('cuda', torch.float16)
with torch.inference_mode():
    out = model.generate(
        input_ids=inputs['input_ids'],
        pixel_values=inputs['pixel_values'],
        max_new_tokens=20,
        do_sample=False,
        num_beams=1,
    )
decoded = processor.batch_decode(out, skip_special_tokens=False)[0]
print('output (zero-shot, ruidoso esperado):', repr(decoded))

vram = torch.cuda.memory_allocated() / 1e9
peak = torch.cuda.max_memory_allocated() / 1e9
print(f'VRAM tras smoke test: actual={vram:.2f} GB - pico={peak:.2f} GB  (objetivo < 12 GB)')


---

**H1 cerrado** si:
- Celda E imprime `params≈230M`
- Celda G imprime `tokenized to ids=[ÚNICO_ID]`
- Celda H ejecuta sin OOM y reporta pico < 12 GB.

Siguiente paso (H2): `DataCollator` que mapee `{image, total, bbox}` → `(pixel_values, input_ids=tag, labels=total<loc>)`.

## I · H2 — Helpers de formato del target

Florence-2 cuantiza bbox a **1000 bins** (`<loc_0>`–`<loc_999>`) sobre la **imagen original** (no la redimensionada por el processor). Fórmula upstream (`processing_florence2.py`):

```
quantized = floor(pixel_coord / (original_dim / 1000)), clamp [0, 999]
```

Target string: `"{total:.2f}<loc_x1><loc_y1><loc_x2><loc_y2>"`. El BOS/EOS los añade el tokenizer.

In [ ]:
NUM_BBOX_BINS = 1000

def quantize_bbox(bbox, image_size, num_bins=NUM_BBOX_BINS):
    """bbox=[x1,y1,x2,y2] píxeles abs sobre imagen original; image_size=(W,H) PIL."""
    W, H = image_size
    x1, y1, x2, y2 = bbox
    bin_w = W / num_bins
    bin_h = H / num_bins
    qx1 = min(int(x1 / bin_w), num_bins - 1)
    qy1 = min(int(y1 / bin_h), num_bins - 1)
    qx2 = min(int(x2 / bin_w), num_bins - 1)
    qy2 = min(int(y2 / bin_h), num_bins - 1)
    return qx1, qy1, qx2, qy2

def format_target(total, bbox, image_size):
    qx1, qy1, qx2, qy2 = quantize_bbox(bbox, image_size)
    return f"{total:.2f}<loc_{qx1}><loc_{qy1}><loc_{qx2}><loc_{qy2}>"

# Sanity unit-test sobre val[0]
sample = val[0]
img = load_image(sample['image_path'])
target_str = format_target(sample['total'], sample['bbox'], img.size)
print('image_size:', img.size, '| bbox:', sample['bbox'], '| total:', sample['total'])
print('target string:', target_str)

## J · `Florence2TotalCollator`

Encoder-decoder (BART-style): se pasa el answer entero como `labels`; los `decoder_input_ids` los genera el modelo internamente (shift right). `pad_token_id` → `-100` para que no contribuya al loss.

In [ ]:
class Florence2TotalCollator:
    def __init__(self, processor, image_loader, prompt=EXTRACT_TAG):
        self.processor = processor
        self.image_loader = image_loader   # callable: image_name -> PIL.Image
        self.prompt = prompt

    def __call__(self, batch):
        images, prompts, answers = [], [], []
        for ex in batch:
            img = self.image_loader(ex['image_path'])
            images.append(img)
            prompts.append(self.prompt)
            answers.append(format_target(ex['total'], ex['bbox'], img.size))

        inputs = self.processor(
            text=prompts,
            images=images,
            return_tensors='pt',
            padding=True,
        )
        labels = self.processor.tokenizer(
            answers,
            return_tensors='pt',
            padding=True,
            add_special_tokens=True,
        ).input_ids
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask'],
            'pixel_values': inputs['pixel_values'],
            'labels': labels,
        }

collator = Florence2TotalCollator(processor, image_loader=load_image)
print('collator ready')

## K · Verificación end-to-end del collator

Sanity check antes de pasar a H3:
1. Shapes correctas
2. Decode de labels reproduce el target string esperado
3. Forward dummy → loss escalar finito

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(train, batch_size=2, collate_fn=collator, shuffle=False)
batch = next(iter(loader))

print('--- shapes ---')
for k, v in batch.items():
    print(f'  {k}: {tuple(v.shape)}  dtype={v.dtype}')

assert batch['pixel_values'].shape[0] == 2
assert batch['pixel_values'].shape[1] == 3
assert batch['labels'].shape[0] == 2

# Decode labels (excluyendo -100)
lbl = batch['labels'][0]
decoded = processor.tokenizer.decode(lbl[lbl != -100], skip_special_tokens=False)
print('\n--- decoded labels[0] ---')
print(repr(decoded))

# Esperado del primer ejemplo del split train
ex0 = train[0]
img0 = load_image(ex0['image_path'])
expected = format_target(ex0['total'], ex0['bbox'], img0.size)
print('expected target:', repr(expected))
assert expected in decoded, f'decoded no contiene el target esperado: {expected}'

# Forward dummy (sin grad)
batch_cuda = {k: v.to('cuda') for k, v in batch.items()}
batch_cuda['pixel_values'] = batch_cuda['pixel_values'].half()
with torch.no_grad():
    out = model(**batch_cuda)
loss = out.loss.item()
print(f'\n--- forward dummy ---\nloss = {loss:.4f}')
assert torch.isfinite(out.loss), 'loss no finita'

vram_peak = torch.cuda.max_memory_allocated() / 1e9
print(f'VRAM pico tras forward dummy: {vram_peak:.2f} GB')
print('\nH2 ✅  collator validado, listo para H3.')

## L · H3 — Preparar modelo para fine-tune

- Conversión a fp32 in-place (`model.float()`): conserva el `resize_token_embeddings` de la celda G y deja a `Trainer(fp16=True)` gestionar AMP con master weights fp32.
- `gradient_checkpointing_enable()` + `use_cache=False` para reducir activaciones a costa de recompute.
- `model.train()` (cell E lo dejó en eval).

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = model.float()
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.train()

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
vram = torch.cuda.memory_allocated() / 1e9
print(f'fp32 weights · trainable≈{n_trainable:.1f}M · VRAM≈{vram:.2f} GB')

## M · Wrapper Dataset para Trainer

`Trainer` necesita un objeto con `__len__` y `__getitem__` que devuelva dicts. Las listas de dicts cargadas en celda F lo cumplen, pero las envolvemos en `torch.utils.data.Dataset` por compatibilidad explícita con la API de Trainer (evita warnings de `column_names`).

In [ ]:
from torch.utils.data import Dataset

class TicketsTotalDataset(Dataset):
    def __init__(self, entries):
        self.entries = entries
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, idx):
        return self.entries[idx]

train_ds = TicketsTotalDataset(train)
val_ds   = TicketsTotalDataset(val)
print(f'train_ds={len(train_ds)}  val_ds={len(val_ds)}')

## N · `TrainingArguments` + `Trainer`

Hiperparámetros (decisiones documentadas en `memory/bot/decisions.md` H3):

- `batch=1`, `grad_accum=4` → batch efectivo 4
- `lr=1e-5`, warmup 10 %, scheduler cosine
- `fp16=True` (AMP con master weights fp32)
- `num_train_epochs=10` con `EarlyStoppingCallback(patience=2)` sobre `eval_loss`
- `eval_strategy="epoch"`, `save_strategy="epoch"`, `load_best_model_at_end=True`
- `save_total_limit=2` → no llenar Drive (cada checkpoint Florence-2 ≈ 1 GB)
- Checkpoints en `CKPT_DIR` (Drive) — Colab corta a 12 h, recovery imprescindible

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

RUN_DIR = os.path.join(CKPT_DIR, 'h3_full_ft')
os.makedirs(RUN_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=RUN_DIR,
    overwrite_output_dir=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    num_train_epochs=10,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    fp16_full_eval=True,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=0,
    label_names=['labels'],
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print('trainer ready · output_dir =', RUN_DIR)

## O · `trainer.train()`

Si Colab corta antes de terminar, re-ejecutar A→N en una sesión nueva y llamar `trainer.train(resume_from_checkpoint=True)` (Trainer detectará el último checkpoint en `RUN_DIR`).

**Criterios de cierre H3:**
- `eval_loss` decreciente al menos 2 épocas consecutivas
- VRAM pico < 14 GB (margen sobre los 16 GB de T4)
- Mejor checkpoint guardado en Drive

In [ ]:
import glob

resume = bool(glob.glob(os.path.join(RUN_DIR, 'checkpoint-*')))
print('resume_from_checkpoint =', resume)

train_result = trainer.train(resume_from_checkpoint=resume)

print('\n--- training summary ---')
print(train_result.metrics)
print(f'VRAM pico: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## P · Guardar best model + processor

Tras `load_best_model_at_end=True`, `model` ya contiene los pesos del mejor checkpoint. Lo guardamos en una carpeta limpia (`best/`) para H4/H5 sin arrastrar los `optimizer.pt` y demás artefactos del Trainer.

In [ ]:
BEST_DIR = os.path.join(CKPT_DIR, 'h3_full_ft_best')
os.makedirs(BEST_DIR, exist_ok=True)

trainer.save_model(BEST_DIR)
processor.save_pretrained(BEST_DIR)
print('best model + processor →', BEST_DIR)
print('contenido:', os.listdir(BEST_DIR))

## Q · Sanity check post-train

Verificar que el `load_best_model_at_end` funcionó correctamente pese al warning de *missing keys* (`embed_tokens.weight` / `lm_head.weight`). Es el bug conocido de Florence-2 con safetensors deduplicando pesos tied — los pesos se reatan en el load, pero confirmamos generando sobre un sample del val.

In [ ]:
model.eval()
model.tie_weights()  # re-atar embed_tokens / lm_head por si el load_best los desató

model_dtype = next(model.parameters()).dtype  # fp32 tras celda L
print('model dtype:', model_dtype)

for i in [0, 3, 7]:
    sample = val[i]
    img = load_image(sample['image_path'])
    inputs = processor(text=EXTRACT_TAG, images=img, return_tensors='pt').to('cuda')
    inputs['pixel_values'] = inputs['pixel_values'].to(model_dtype)
    with torch.inference_mode():
        out = model.generate(
            input_ids=inputs['input_ids'],
            pixel_values=inputs['pixel_values'],
            max_new_tokens=20,
            do_sample=False,
            num_beams=1,
        )
    pred = processor.batch_decode(out, skip_special_tokens=False)[0]
    expected = format_target(sample['total'], sample['bbox'], img.size)
    print(f"val[{i}] {sample['image_path']}")
    print(f"  GT      : {expected}")
    print(f"  pred    : {pred}\n")

model.train()  # reactivar para futuras ejecuciones (no afecta si ya cerraste H3)

## R · H4 — Eval cuantitativo sobre el split `test` (14 imgs)

Métricas:
- **Total exact** : pred == GT
- **Total ±0.01** : tolerancia 1 céntimo (cubre redondeos al format `{:.2f}`)
- **IoU bbox** : sobre coordenadas de-cuantizadas a píxeles de la imagen original
- **Mal-formado** : el output no parsea como `{float}<loc_x1><loc_y1><loc_x2><loc_y2>`

De-cuantización: midpoint del bin → `pixel = (loc + 0.5) * (dim / 1000)`. Compensa el `floor` del quantize.

In [ ]:
import re

OUTPUT_RE = re.compile(
    r'(?P<total>-?\d+(?:\.\d+)?)<loc_(?P<x1>\d+)><loc_(?P<y1>\d+)><loc_(?P<x2>\d+)><loc_(?P<y2>\d+)>'
)

def parse_output(text):
    """Devuelve (total: float, bbox_loc: (x1,y1,x2,y2)) o None si mal-formado."""
    m = OUTPUT_RE.search(text)
    if not m:
        return None
    try:
        total = float(m.group('total'))
    except ValueError:
        return None
    loc = tuple(int(m.group(k)) for k in ('x1', 'y1', 'x2', 'y2'))
    if not all(0 <= v <= 999 for v in loc):
        return None
    return total, loc

def dequantize_bbox(loc, image_size, num_bins=NUM_BBOX_BINS):
    """midpoint del bin → píxeles sobre imagen original."""
    W, H = image_size
    x1, y1, x2, y2 = loc
    return (
        (x1 + 0.5) * (W / num_bins),
        (y1 + 0.5) * (H / num_bins),
        (x2 + 0.5) * (W / num_bins),
        (y2 + 0.5) * (H / num_bins),
    )

def iou(boxA, boxB):
    """boxes en formato (x1,y1,x2,y2). Devuelve IoU ∈ [0,1]."""
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0.0, xB - xA) * max(0.0, yB - yA)
    areaA = max(0.0, boxA[2] - boxA[0]) * max(0.0, boxA[3] - boxA[1])
    areaB = max(0.0, boxB[2] - boxB[0]) * max(0.0, boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

# Sanity sobre val[0]
sample_pred = '</s><s>19.55<loc_614><loc_558><loc_682><loc_594></s>'
print('parse:', parse_output(sample_pred))

## S · Loop de inferencia + métricas sobre `test`

In [ ]:
model.eval()
model.tie_weights()
model_dtype = next(model.parameters()).dtype

results = []
for sample in test:
    img = load_image(sample['image_path'])
    inputs = processor(text=EXTRACT_TAG, images=img, return_tensors='pt').to('cuda')
    inputs['pixel_values'] = inputs['pixel_values'].to(model_dtype)
    with torch.inference_mode():
        out = model.generate(
            input_ids=inputs['input_ids'],
            pixel_values=inputs['pixel_values'],
            max_new_tokens=20,
            do_sample=False,
            num_beams=1,
        )
    pred_text = processor.batch_decode(out, skip_special_tokens=False)[0]
    parsed = parse_output(pred_text)

    gt_total = float(sample['total'])
    gt_bbox_px = tuple(sample['bbox'])  # ya en píxeles abs

    if parsed is None:
        results.append({
            'image': sample['image_path'],
            'gt_total': gt_total,
            'pred_total': None,
            'exact': False,
            'within_001': False,
            'iou': 0.0,
            'malformed': True,
            'pred_raw': pred_text,
        })
        continue

    pred_total, pred_loc = parsed
    pred_bbox_px = dequantize_bbox(pred_loc, img.size)
    results.append({
        'image': sample['image_path'],
        'gt_total': gt_total,
        'pred_total': pred_total,
        'exact': pred_total == gt_total,
        'within_001': abs(pred_total - gt_total) <= 0.01,
        'iou': iou(pred_bbox_px, gt_bbox_px),
        'malformed': False,
        'pred_raw': pred_text,
    })

print(f'eval done · {len(results)} muestras')

## T · Tabla por muestra + agregados

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print('=== por muestra ===')
print(df[['image', 'gt_total', 'pred_total', 'exact', 'within_001', 'iou', 'malformed']]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

n = len(df)
agg = {
    'n_test'        : n,
    'malformed'     : int(df['malformed'].sum()),
    'total_exact'   : int(df['exact'].sum()),
    'total_±0.01'   : int(df['within_001'].sum()),
    'iou_mean'      : float(df['iou'].mean()),
    'iou_>=0.5'     : int((df['iou'] >= 0.5).sum()),
    'iou_>=0.7'     : int((df['iou'] >= 0.7).sum()),
}
print('\n=== agregados ===')
for k, v in agg.items():
    if isinstance(v, float):
        print(f'  {k:14s}: {v:.3f}')
    else:
        pct = f' ({100*v/n:.1f}%)' if k != 'n_test' else ''
        print(f'  {k:14s}: {v}{pct}')

# Mostrar fallos para analizar (no-exactos o IoU bajo)
fails = df[(~df['within_001']) | (df['iou'] < 0.5)]
if len(fails):
    print(f'\n=== fallos ({len(fails)}) ===')
    for _, r in fails.iterrows():
        print(f"{r['image']}  gt={r['gt_total']}  pred={r['pred_total']}  iou={r['iou']:.2f}  raw={r['pred_raw']!r}")

## U · Visualización pred vs GT (auditoría de bbox del holdout)

GT en **verde**, pred en **rojo**. Útil para distinguir entre:
- **Modelo falla**: bbox rojo lejos del total real
- **GT mal anotado en H0**: bbox rojo sobre el total real, verde en otra zona

In [ ]:
from matplotlib.patches import Rectangle

n = len(results)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*5))
axes = axes.flatten()

for ax, r, sample in zip(axes, results, test):
    img = load_image(sample['image_path'])
    ax.imshow(img)

    # GT bbox (verde)
    gx1, gy1, gx2, gy2 = sample['bbox']
    ax.add_patch(Rectangle((gx1, gy1), gx2-gx1, gy2-gy1,
                           fill=False, edgecolor='lime', linewidth=2, label='GT'))

    # Pred bbox (rojo) si parseable
    if not r['malformed']:
        parsed = parse_output(r['pred_raw'])
        if parsed:
            _, ploc = parsed
            px1, py1, px2, py2 = dequantize_bbox(ploc, img.size)
            ax.add_patch(Rectangle((px1, py1), px2-px1, py2-py1,
                                   fill=False, edgecolor='red', linewidth=2, label='pred'))

    title = (f"{sample['image_path']}\n"
             f"GT={r['gt_total']}  pred={r['pred_total']}  "
             f"IoU={r['iou']:.2f}")
    ax.set_title(title, fontsize=9)
    ax.axis('off')

for ax in axes[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Diagnóstico: imprimir coords crudas de los IoU=0 con total OK
suspect = [r for r in results if r['iou'] == 0.0 and r['within_001']]
for r in suspect:
    sample = next(s for s in test if s['image_path'] == r['image'])
    img = load_image(sample['image_path'])
    parsed = parse_output(r['pred_raw'])
    pred_loc = parsed[1] if parsed else None
    pred_px = dequantize_bbox(pred_loc, img.size) if parsed else None

    print(f"\n--- {r['image']} (img.size={img.size}) ---")
    print(f"  GT bbox (raw)         : {sample['bbox']}  type={type(sample['bbox']).__name__}")
    print(f"  pred loc tokens       : {pred_loc}")
    print(f"  pred bbox dequantized : {tuple(round(v,1) for v in pred_px) if pred_px else None}")
    if pred_px:
        # IoU a mano
        ax_, ay_ = max(pred_px[0], sample['bbox'][0]), max(pred_px[1], sample['bbox'][1])
        bx_, by_ = min(pred_px[2], sample['bbox'][2]), min(pred_px[3], sample['bbox'][3])
        print(f"  intersect (xA,yA,xB,yB): {ax_:.1f},{ay_:.1f},{bx_:.1f},{by_:.1f}  → "
              f"w={bx_-ax_:.1f}, h={by_-ay_:.1f}")

## V · Inferencia sobre una imagen externa (test out-of-distribution)

Sube tu ticket a Colab (panel izquierdo → 📁 → Upload) y ajusta `MY_IMAGE` con el path. Por defecto busca `/content/img.png`.

In [ ]:
MY_IMAGE = '/content/img.png'  # ajusta si subes con otro nombre

assert os.path.exists(MY_IMAGE), f'no encuentro {MY_IMAGE} — súbelo a Colab primero'

img = Image.open(MY_IMAGE).convert('RGB')
print(f'imagen: {MY_IMAGE} · size={img.size}')

model.eval()
model.tie_weights()
model_dtype = next(model.parameters()).dtype

inputs = processor(text=EXTRACT_TAG, images=img, return_tensors='pt').to('cuda')
inputs['pixel_values'] = inputs['pixel_values'].to(model_dtype)
with torch.inference_mode():
    out = model.generate(
        input_ids=inputs['input_ids'],
        pixel_values=inputs['pixel_values'],
        max_new_tokens=20,
        do_sample=False,
        num_beams=1,
    )
pred_text = processor.batch_decode(out, skip_special_tokens=False)[0]
print(f'output crudo: {pred_text!r}')

parsed = parse_output(pred_text)
if parsed is None:
    print('❌ output mal-formado')
else:
    pred_total, pred_loc = parsed
    pred_px = dequantize_bbox(pred_loc, img.size)
    print(f'\n✅ total predicho   : {pred_total}')
    print(f'   bbox loc tokens : {pred_loc}')
    print(f'   bbox píxeles    : ({pred_px[0]:.0f}, {pred_px[1]:.0f}, {pred_px[2]:.0f}, {pred_px[3]:.0f})')

    fig, ax = plt.subplots(figsize=(8, 10))
    ax.imshow(img)
    px1, py1, px2, py2 = pred_px
    ax.add_patch(Rectangle((px1, py1), px2-px1, py2-py1,
                           fill=False, edgecolor='red', linewidth=3))
    ax.set_title(f'pred total = {pred_total}', fontsize=12)
    ax.axis('off')
    plt.show()